# MVA Track 2 · NB15

**Produkty readthrough: L737W/C/R, panel wszystkich podstawień i opcjonalny lokalny ESM-2.**

Wersja 1.0.0 · CPU · 29.09.2026

Wymaga ukończonego NB14. Nie klasyfikuje funkcjonalności białka na podstawie ESM. Domyślnie próbuje uruchomić ESM-2 CPU.

Wyniki są przedkliniczne i eksploracyjne. Notebook nie generuje rekomendacji leczenia.
**Public release note:** outputs are cleared and storage is controlled by `MVA_READTHROUGH_ROOT`; no Google Drive mount is required.


## 1. Konfiguracja

In [ ]:
# Konfiguracja — domyślnie wystarczy „Uruchom wszystko”.
NOTEBOOK = 'NB15'
PROJECT_FOLDER = "MVA_TRACK2_STRENGTHENING_v1_0_0/READTHROUGH_EXTENSION_v1_0_0"
RUN_ESM = True  # tylko NB15; False = wyłącznie rdzeń offline
MODEL_REVISION = ""  # pusty = ustal i zachowaj dokładny commit modelu przy pierwszym uruchomieniu
NB14_RUN_OVERRIDE = ""  # opcjonalnie pełna ścieżka do zgodnego katalogu wyniku
NB15_RUN_OVERRIDE = ""  # opcjonalnie pełna ścieżka do zgodnego katalogu wyniku

# Zmienne LOCAL_* służą wyłącznie do jawnych testów poza Colabem; nie włączaj ich w Colabie.
LOCAL_TEST_MODE = globals().get("LOCAL_TEST_MODE", False)
LOCAL_PROJECT_ROOT = globals().get("LOCAL_PROJECT_ROOT", "")
SKIP_ESM_FOR_TEST = globals().get("SKIP_ESM_FOR_TEST", False)


In [ ]:
from pathlib import Path
import os, sys, json, hashlib, gzip, base64, importlib.util, tempfile

# Public/reproducible root. Set MVA_READTHROUGH_ROOT to a persistent directory if desired.
PROJECT_ROOT = Path(os.environ.get("MVA_READTHROUGH_ROOT", "/content/READTHROUGH_EXTENSION_v1_0_0")).expanduser().resolve()
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
print("Wyniki będą zapisane w:", PROJECT_ROOT)


## 2. Zamrożone referencje i kod
Rdzeń korzysta wyłącznie ze standardowej biblioteki Pythona. Nie wymaga instalacji pakietów. NB15 opcjonalnie instaluje ESM w osobnym środowisku CPU; pierwsze uruchomienie wymaga internetu i pobiera kilkaset MB. Log postępu pojawia się co około 45 s.

In [ ]:
# Dane publiczne oraz pełny, audytowalny kod źródłowy (sprawdź pliki .py w pakiecie).
EMBEDDED = {'BUB1B_NM_001211.6.gb': {'sha256': 'e6162e0c229509602c9aaa77d1c4ce20932694768e32a92f4c9f5cdb4dea3c99', 'gzip_base64': 'H4sIAAAAAAACA+1cbXPaWLL+7l+hyny4mSogekEvOJUPMghMEIqQCF7v1lTqWCiybgB5QWTGqf3x9+k+kmM5JmSyWzM33qESI6Rz+nQ/3f10H8nYf9N/GyvyFUzfqaqma5rSfBmW1VOubuhwHQUuva/yTSq2OAijsaJZ7ddvg7au6tbJwBuOg/F8/CZQlPNiXSg7cZOnm51y9vZMU9Z5WZR5oiTXafLhpsg3pbJLt5D1orzepsUGR8qHfCN2qXJ2cl+F5zT97OcWK9A5cft9L455kc9anyy8qDp372zHOpl4lxdvogGbGaXv4/SfL5WpG3hKnK7SpOycxG/eRn1PrtRQ+vn1fi02P0OVN9HIDcbxtDmgoaO3/yC2t0UpID0txacCB/3rYrsUdKq/FZucjxbptkyvtnzs7UvoUOzKYp2/bEibivVarHI55hog8ZHYJtfFpiyyVb5Ndy+VcJuvRUlH5+JmVWy31/nmgaA+FqrOk+54X4qUD4vOSeQNvcgL2Hi4/fkVoN/hqCzY62S5+3Z+/iYi8OLkWrx/n25bfmfRaSnn+9YUb7NcbFojHEyLYtOadOhcKFYtF+8xXNoKOq2GQr4or8Ua10MaCWvEplAWIocZkNxS5juRtwYkUJTlDjIDlunnCctvCjsTiKXVEkpFGNNPd5v8Ay89Evtta0zz0tac3xBcv6ZXGHkhNRRbGtiQFu5X+bJoeWzWpnWJ979fi6L1N5J9vd+KXavPsjYf8vTXPPnUes1X0g0dNES92RSkLDwEK9icv4ssbZHNHlT+mG7hUlYwhptZ5XOxvYF6rzsXD2SNbrO8FTNeQ8QKe+A1e6AskYnQl3Dz95slzMvwURGbpTJeph8wdN6BrPl47ssAn+5XZb4ulmKlJOlqpazFzU4R+Ke8LyBAlHmxweFW2ZXbfVLC5hVLe7/fJHRNrBqqZekGMZVQJrxGFgWuT8knMC9VrK6uPHdUy0Le6rre1g1NeQ6WMCmslPDt2dQb4KCr6qqpml2cjLypG004mbbAbb8+/YeU1VFonvImKV9aXQtCHevnU0+zOsqyyE8bGmlqR1MN58Wuq5mO1ca0ttqzuk77t44STseDU6WL6z3N1H9phL9+LPwRCZusdc6hiNBgVPw9TjTw7SMJ05WEsXgvaY9GMoEpv+bltYJ0VpaIgFVxs05BgXT5Zltkm2KX7zCpYU66WRbrtESwwGFik6TbBtRxkoPTbhStqzzXgLNmq06XUe42UTZ6atexur0GyqN0k0bj4el3at1Q9L4Fj2jd5BrjGNiDHFhfch4XjSSDtfL9dU7ueC1TFEd/J+oocPC3zhe6cRK/bjqKTLtS8vWNyLc7qkjrgnhW3NyiJm12eZl/zMtb5WMuFFBtvivJDCjbEAw23BY3JZudb5TVfpMpYomcSMDUOVJDNLzVp4wbpCBAGAjbTeW5DaeZCO0DLjMs03jcZX+Q/k2/dY/5rS8+ihU8XqbgNmKlOF99FKBprEPk2e8QFfYLsC/oftAZ0qft/lNrAZKP8WGZNhR0N8stFGqdddzOGS4HYKAPAoKGxJJKJK7SDWrsrgQ/nmPEF66/p1CE680Q2F9FMs6T2wREqiDq8/V6vynS326g347SAadvUKLXxfbmGq0Lo7MWX4Z/sbrlMcV+9wBCSgjUXeJUUP1HNAlKtsL8JmmGiIpihfDeUc1ISkU3DeSz2e1Z2qPR4SA6tF73QHT8ftMa5twz87tMa4aNeSxsAgQxhwCVw6s9ymHM9W2HXmDDxXFSrK/EVnxAj4V+Y8DF0BcfdtfrvHbsPe2h9VRsNgLVb9L0uWe091f5P/fID8R7nkErVNWzv13YSJNsv6Ju6q5PJU6rwbq6bSyBpipLISOroKaSeSe4vU6XaPbSJcI5Q/wysX7JBFP428/fp8zijgoG18EGhi0J3PjC36rjHGKD7zWrYdPV7XeY1fS0dZTYxcd0c1NsS9nnKMN0u4G7QAZeJ5JN0DanRpB4IkjFKkVz6X6Z16Oi2F4LuL7h3TmK1Bphmj621UCvkipX+yvtqoV2Z1OglvFH5b1Y56ume9fpmlpFBbxYJQysBrRCdk2UVGl7md6g0FExxC5h86Ayj6rOSDHNqjRrRlvTbHzo9XpN3/Ycp6eqZgNJ+yjVXnPjTU0mdeZoZ886c9mWXKJGzjsPCh7GE69gK/HprtBTUceGKy0L7CvQsl3lmyU5H9UbiUcj+l4Qtr0GNlt0AWJFBQPCBHeFu889wn1xCLQSffdOShl+4cRrFLFIa8D2WqbGWY7c0LqGhK7ba1sGA+c8AM62jK6uNoBzjgE3F7fYKqFgMMeIul+eJhNsQVvDLyOKd4HKNYhyVWT7lDuk/ZVBwbFNkRpbxAYlyj3TG2auQJire6ifUdDRkgIstTRe0Oc291/pssas2gd/BRu9CisE1aPQWJbqmE4Dmt7R7ER5XWKNnLdTZ+kGm5ecd2VRut5/+iQ3KtNim2NHQmw92q/A9zuAGT3cmaWbPNvICfXkkUQ6KCjFp02gx5RI+fs8uYOpzlHK23Y7jidauw1gpDeow0S4rfJqI9PciyNnC8pYGLm7pjsV2M0ghxtoAkbqmuj95vp2x5W3X1CNVPSupTw3AK7jaG3HsR/HV3MM1WpuodWj7dF1vlrV5SvdZGIPlqE95eVedrd5liFhgBR9usBg2gG/7jzc907F9kOBnPsE5IkpJ/nm0wqiJiBVxnhRZOlqR5HUOnu4AyyFzFku3o/SJHeDEmjZvj+6wTN6iEHLsgkpQ1Xbhvp4kpq6Zuj2Sf/NdOoFc1Yi8hZj7wKXAV/szU6RZ5xNSbFdKtfYj16lKdbG9pNyAoUp6J+N4UXx/n2HcvIBIaGPTam93SEd+eBXiFiC6j5Sam6LteL2sRPs6r2O1WwVHmzaEZsfYQs2rKrWA59BqTuZ2LNzJ7BNb1YigeDPN5nMrwiN9+s1GqPKRC5CEFeg5CHCq7td+eZjsfooS8zuBiS8Su+5pNmyVxtxhuGOLe4gq7hGxv5DPv6iaV2JW9JiW6xIBx6fb67zq/x+gRAbcQP5aRuLYZOB+tAQkhRrdJG/vaCaWOyw7VOeu2H/Rf9nauaxANUTEoOIS0vO60reF+pg27Lf0nAsdJNugQAtyCJ3aDjSTDYcypg3PTDxG6H6jA7f4yBLEdm3xNlrzoJqi6r8Awt/zJcccQ1J8rYhNp37FSJDdX75ir/D/dWqZrEA3jlVpOeljDrG802y2ssY2O2vKmTKB4F981nUDhcFdt3kxY8iX4kr2E1Fp6yjqqOEq5Rg3aUpA07N4YM84aURGCJJQIdo5pe5vKnTWOorxv30k0cAAa72QJSiHc/daP7TT40xc3Tou2Sb35TgYYCAAEFfIQE5PVXcoabaes/uaC0cq5ZjW2pHU/7h9d+cqngZhv5LE/zARQ5Su7ElHBovyNshYoDFTlAYAswb6i5hGyplNeVE+fZX7E49V+tZpt7rte4+WZaj/x4pn43pOr98FUAvGDyA78FwGTZttyy3+dUebfyjmPP9bA8Ztr6ie3plcl3D4wXxnDRRdcc2e05H017QufD+OfuRYK/ujitwJPWMgmRRbVsikWvMK/Zpg804y+8cf9SGL81GdQh9b+4FXhyfVqRSEmtwLBv/QxW/cxJGY+x9LqsaQqXjXRy6wefkk9ffjQeoNOPh2IvuTt0fR4s17xq2NZT55utzybg7Zfcstde2e3bPbE53nLZhaMemO4bZM9qOYRtWYzpCvm30jk93LFVHP2JZWnN6T2+bhnV0ek+17bbT0/XmdNOw27ZhHJveM1XNbnMxb0y3jW67px6drqk6TrXx1tWbAnoqTmuac1SAYatmm2/e9proa1qvjTqsHhPQVXtwtNrVHmQz1fC21u0eE6CZaKmhq2lo3aaAbhcCTNM4JsDBIAhwdKuZczgNEOhB2zEBtkNoYcP4AAPLVtuarR0V0OvatGWAH9WmAFsnAc4xDHS11zUZbfVBDtgOZZFjHROgaTq7Sze6D5PIbuuapR0T0LV1YKB30Wk2BGCu3tb1nnlMgK0hEfBmaE0MMNdq613jqAmOZhMGjqE1McBcmICEOirAMlUSYJsPNDANp61bzjEBBqVBG28ArSnAAhVBMfWoAMfWIQB9btMLZBQ6eVU/JsBSDQsCLF1thjLmGm3o1zsqoNcjATYooSlA07CXOJ4L4AFiDgPl2TgZeu78beTFDRryC9nRvJjt0RS/z+U+Bq9dsd+io/8sq9OhBR8v8S+KbSY2+W796tn9h7/PDoxeF6t35e1N+uoZPaw+NGp59e437FtePSsFeqRT1BXr0NDPbfCrZ5p5cFlxQ5f/qZkdrRrDew3lW62k0a+e8ROfZ18Z8m53uyk2twCDblu4L+WDiJSeZvPdnJfVXR16xj4w/JePi5ouXIygPf2htTZoLaQ6v+vXBo7iTV3xeHBqq9rRoeejoH/KPzSt2zs6fDqenqJAo0hXQ7n7beKPrPsx4M831Z761TOkTrahZ36n8Q0dnuodrVPb2B80Uw61EVFmPKgN/2/N9PrvNnu61fvqmd7BlkTrHAlIWu4FKXPght3LOlofl3IvhKuIlforV/sl99D7TbUB/3IT+vk+fbr5lK/zpfiEfTvda6puTSp3AGlnrUey5nFpB1LpIBsVy2LzbleKbflKOzAGkCz3SQkGPJ657Qq/x0XVaU00QNY9O7wiCXmXL189C0K+LaPqHfNozvYRv6f0g9oZo6P9f6KPL4by/ko+padAdBeTiTcauX7sudPY90YD78Lz44kXLGahH81G2PjEc382ch+X57szL3b7QYAxs9kkcofepTeOhpfzUTAI/cHiYhBdjuOLuTcLLsPZaDTx4oAk+o/L8yJ34blYz5tEl5fxIIyGfuBf+BN/FPn9wIPI6WV86Z8Hs9F4tIix/pDku94BeZeeG3nBMJq4A3c8nHmjMdR0IceL/Fk8O49mw5kbLeJoBp1c35t49FoMD8iL40U4i6C/6/mTeDKaTLCRdsPxOFoQihM3jGdBNPJnQTgMZ7PpLAjiaDxfDAcH9AvcgediN06gh3OAfjF2w3AaRi584PkzdxReBPNRBN+cR2E0CuZu7I/dRfi4PHfhh/FwHl4uPA+azcLFdB72J2MvjMfB+diP59EkhBPg6NCfRYtZfD6bubHnTR6XN/Em0+llH2/jS3e0GGG7PPTgYBcQRZOJP/FmkQfl/XnswmWuN51NZmPPmx7Aj6eMZ/P5bOZF89EAb958Gs4n3nw+8WdjqBLPJuNwNJ37McDuzxZBvw8Xzg/4A44Ixhczbxaewx0h3DOMx8OBN/QR0pNJMInD0B2EYbRAqERR6AOhyRyrPC5vPI8Db7CIw8GiDyHzEZCDIMTOeD4aRsFiPu6HQegN5v3BEJ53o+EinofD8wP2Inv8yQA+GYQIOD/0vAGSghSA6XO3P5qPL2OEXjwGMjHiyBvEAHR+fkC/eDSMR3HsxgsETYxZfYAWehPf8+eBB7FBOJ/F4UU/nl1G0cxHkGJRHxMOxIvn97FmhIiDjDHCf+z5I0Bw2R9PIu/SH/dxPPGHFws3jAI42Z8vxpNFfEAeAi68GCAlA588HfkQNRycD/tx/3J2ORuM+uPFxfkM1/tDUMbA92fnsXc5np8fyA/kjj8eXwaIsPHCm/rnSN7x4nw0AFpR3x/7URCNzwfh5aAfTIJghhwcLwbD+PIAfouBj6iHD7A8wIyQcmPQHmLOd4N+HIeXMwwZjsbuwHfPfd8fTrxzf7YYPi7vYjCKhxdw3SwYx6CEwcjzL4LJcLiIxn7gBkju+SJe+DjtIjmCEZhgPh8i7Q7waTBx/YvJYjRBRoVgFH84q6h8ne+Sd+9T+diEfo/GQZdkqtaP1IrTTXmRccvDT1TebvIQVXdy9iL+Nd/t2vRBef7GUk3L6hg/H1gtSjOUr1Ml2CcrejDXeDa6Q3cpVp0DaqTVjdNX9c1V3TQP4IvdVKdjq/Z/Mb4p3eGnX8ihhwT3HtXLJ/WTwNe+HWer969wfzVNl6ea3XM0zTiEu6PrnY7Tdf57cR+0r4rfOl8HUbdNtedYWusAb8oZtt3t/iscnJ2afv/iAN49Ve10eqr+I+EdWG2RpOXtanW7Q//foqe7Yd8dflc49rqqjYB8qTzw4uOyjrlWkYrVZFSmB2DXdJP28rpp/EjAh9fF7gb/eQP2rWjLKKzi1tAQtZbxn0L7Riq0/TbEu0DcMLT/YmrJd8V2mW7T5aP8Ylr/it3p6bS4ygdn7dVXsOyaJrDsmvaTj94mV6j04Mb+Ftb98+Pd1Bz4yOzpf8X7vx3vtt0DlqgTf8X71+Nd66qW1dNbX02Gx8X9Z1z/ezLEcWy6n/5jZUgfm56P9EUw6noSsbuh36syvsu9lo4+0rb+U+SUVJp9FfOeg34TP82/MP+jMEdSolrjp/1Xr/nH1F5ds1RC3NL/qhdfrRfwkEVfrWwduNjTerb1O8qPqloO/P5DlB8d/QQFiaP+sEEid97+5DtvBNmOaerOn5Gfehe7b/z8a/f9RyFu6hYQh7t/QMTvnjT/m/Fu2BY2En8C+uDkbqeDn9Zf6P8J6Os2Yh8/f+jY/y7CqerxnwN6j0D/L92y/wGIf/G7aY5Dv7elPa3fTntopWHosLL31K3swUrTsJ62laZhdzq2YTxtK22jS0/4nriVPZWe7mia87TNpC+q0HNDTX3idtJ3Yjv0hZonbmeXdqL0vZ8nbqfJT19/mO8OfLedlk33t23tqdtp62yn89Tz03b4OxfOE2+F5LMw+urc07aT7snTnb+e+cTt1Ht0v6371Ft4+rok3Vc07Cdup2nQwwLLeep2WnQXgb59+sTtdAz6LqX6o/zi5XffSFBV+jafpj7xfoi+cfwjfTf2W+2s/vRfsb2t7LS6dFvVMrsHBH+e8C5Zid3u1TP644/uO/mL+e/qP4X07Ee6K3ud/ibW6fZUcd2569ZfyK7tKtMamh/ri9Fr8b/FVlpR3dZ9E41H4+ovnNxN1ZSszLIsKctMKBkdJwneFHwuS4ErOFsmOF+KJFNKXMUVnFFoSkKn7/7Yk6UpSYnZNCtR8BMjExIgMiEEZia0AkmneXSIMzRaocskOau/FKvpGmbhaoZVaZagqzij8DmBkXy2ZHkJKy5omUyhH/Qqa700h/Wi6SUrg1WxfgJZmM4mQoOSpbK2cih0Ill0Fqaf1P0I9CJdgA2tlfBiCZlQVlKFkjA0dIFtoFXoLOtHepzU3Am9GEGBawoWKeU8sjFhzMgwHJbsHFGykviEFegcZohaFmOf8WUMBbAlfwTKNJmO2Lskg1egiwy+wisldL6S1dUZL7IMoySeNLKE5SXbKMhGLE8BAr3oItvBY8ngO7y6DuHF8wlPciZPgyxyRkl6KmwMr6iUUm/goLCljG4ly+xqbA07H0NpVXIEqVhKL1GsEo5lBXgi/QPoKGZpciXLIuzlwFLOSiQ2CruRQM8U6TBRBQ3ZmpENgr0Pm2pZFttIOJZZIpWRfpShiE8JzeLwxAByYsIocXxl9/GyOe5LTgwyISEvsQCyBLqSx8gFZcJBJThoMgZB2iPu9LIl9tU/hUMdRyXlNkc4ocx+I22kc9h9WLdkj4o7vRxgXwrpacIg41CjFBCsqYwkin7OA1KcqaHilITde1LfyiUbMzkas2QAVirKsxTsCcd9wh4RElwCgTyCf7Usi2KC/QQzFHYxGVfWqQsrOLc5phJKMumpjAkuoSyubNRUjnsawFksaUImDkDhl2ACTCTBJRIPwYvRQmROLYs4hzktkQEqCNeaFiWHKncnyUZmXpLFGlL81HpphD3pwUlGf5kt4SMCXEYKuTSR0JWkIhMBsxprRUFUydIZezmH9SbLBIvlfCNF2fuUDZTmpCHpKySVsa9qWZaUJWlHyapIIz/KxCaPMLNTegkGKanoh2DPPnOhZnDcZ1W0KqI6ks6jfMpYg/q0UrGArE7M99C+lgXsKd6SijYlZxJHy5imGFGoTtWLEZdXESzV/ezHblfjJKT0Lxkvgi6T1kj3VwxalSRRxSoHYMYm1LJMxl6ClEjAeZ6MpIT5X6mSiTOLg4ODuc6xO71Mxp7plMNHSM1lPpIswdmQSKfiUNblMikrbcUd52iWrt1VVZrFtYDMVCS1UJ4QkbDQUiYOh1Bl7v1YtQh7WfUJWlm6sooLmZBJFrMt6atU8UveZX9xDFeybIr7Kp3ZzyXXRxnsmYRPkRTNeHHMJZUbSpmvtV4OsM/Y85RGklIkLSYyRRNmBC5CMgWSqmnhwp7whVoWYV9W8UkFlpW+q7WlDFsh63MmD5nyyQ0USsRGlayezrKY69jPSc1UVUyVjKKQXiBHC9k0CZm7vF4ti/iegz7h6pUlsmJIvpdOVaq8KBNZQJlMJXRsUSVLV6nPYcMkbcqM5vgijuYqq9QZlHHdlnDIdkC2H5UsTfK9qGpaVhGc7ABl5yFZjzHnAOSRMuqErES1LOL7hMsM4SW9wLGayA6QbRSSDFlx2aYmtFgVP7Veus79VyL7VFnCOYSUUvIUM7vsyeiwQoBpsZRNU12HdJ17TM4gNqGO0IRWEGwQe5dLCucQu5p9zvUouatDusG1tqKqSkBdsgSzT8J1m88R0bBz72KCga716qqa7Nll4ZcNovSjqD8oVaNVVk1VVtfSRDJjrVfX0iTrkT53zaBclmOdKYOTmyifW30u8XdtSlLzl25S3LP4pG4sud/kSJJpyO058wMV0IwjmxvHsuKR2kbT4Z5JSuA2hrtzUVEwM4hsz+VhIgupzDfZEN/pZXHcZ7KdLWU/wVFaDSU9ZVVlDzJpMXeWSlKx1x1eNsU9s5vgsswgkTlMm9J0XiGrgpmlJ0xwpez77vSyLfKjrCcyW5jwq4pTcsuiyNgioJjrZPMru0VuQitZjq7VDCq7XBqZsAkUKKWsXnJXklX0I7tPpaxo704vx5F6SU6p26esKpWyf+dIyrhNqBs0kVQUS11sVbf1HvN9kkmCUWQdpZyrtiCyL6zOZZILS05qpWIqiD2pb0sx55SSi6pNUFn1chwTXLcTydIlbxA4z+te7XNPbqiWlCXJo9p1SjYuq+aHDpl7hJCOlttNpgRZJypZtK+VAcENXFW8mGJlWWWQeDZvJhKmFJaVJUkFbi2LOUeyWsZ6ZdXOJ6s7EsExkTHoSikrBvX3peyyszomDN7XJrLlF3KPzC2W5InqPoBMMuI+yYV820CRe6jyLr4Mg7EvZdMk98ilJIdEsh7HV8K0VBcqDl2OulK2r7Usi3OIFZP9PXNOIgOU85wTh8ODDZN1SO5huKzWvGrQvlbCJZJqE8R7xkyR+2LJE9SW8nrSjxXnyDQu6x7T6Fa1lqld3sqQ+4lqe8jeFfXOWua2bHq4/+JZtSyTsRc1BkKKqhhBMkZZ7fk4VnlXJPfbMu1oRCWL9rVc7Llj4C2bYJuZVzmWEqVq+aVe3NYx9YuKSWrsaV8rO0BcOHnx4uTk/wD7j3vGDHAAAA=='}, 'BUB1B_O60566.fasta': {'sha256': '4ac6895daf90501d8398d17ded42f46d78b9db777f5084facc875c2926496d1b', 'gzip_base64': 'H4sIAAAAAAACAx2Ty24cOQxF9/UV9QODJBiMgVl4AFqiVIJYMh+KKloNHKOBNILpNuxe5uOHndrUQyJZ99yrfz7efj0/fP7r4eHX09enL0//bl93aOt+vl1v59f19cfp9efb9Xy5rR+n9/Pl9On24/10vfjTH2/v19vpfFl/ni8vH6f1Xr5+P91e1md73K7/XdePl7fz6fKxPn97/Pvh88Oa2+PvGSvj45fVxuOfyw4wasWcgQxhN8Ic8UCyim0Ik0rWslsn8R0gaBBa8zeRqgsknFg0zZ5bZIrjiDqLHR2lTZacK1q7FxMqDARvglXntMi6JGp0UKWsFBp69T5t0tYklzzMZ6V7K0AfAYotaYUIJQnmIotU8BJUEpNNJQnoMBWfBIQV79dIaDZY1OcDUrWaa+2gsHApOu6KK7BJ00zSOLHILq2Zlj5SxAYRwTrcUXB3FEcB5n1hBUeDJJD5aD2rI9uUNbcORgUGwyC21HkORJ8nPPbOoRbkxUrbClnXys7GUTOJDrFNBAzRjaj7PoPfyoQ8MiZL6ITB1SxaK1UURf8n6gaOEnCXKgVxx9+LRXoXQe05+g37zr1i75UWKT7ApBbOeydzMkFGC8HRdoeNrRyCwptTYqeWrDiDRB6KWpdWjRkisw73RZXJNdbuDUu3hnEYxxF8f8+u0mvcqNJz0jb6UgI3xthDTA4fNA3rnDb0VFGNjiqy+0iMGD1B967+7x1C7mUu5pZacW3mpmE0F983s5wsm4ENd8h8PZAUxkpIvbkebNzFFj6CyVQVcvd9AHmFswveSHln3148KwUpu4YZSlWcVII/O7B0DGBt94I+Sh1mbiQf0VPZ6M5ayatS3FKwMGVKzKGMYxNfX0LyIxKJZDOcpbvYPqiU2dy4MnCnzfNbxpajC9NAhbRp2SLPGJx2a+LhLCMmmzYieUQcjfd04epZLH5K3UqCFsx4im9JuUAkWDYiShU3kpGOmC0dzk5aMT8BMSMdraY0tFCD5gHvwwb5Z/AktSV78HtPnkdqFeioI1dPGvtRoSTL/8U9wHarBAAA'}, 'INPUT_MANIFEST.json': {'sha256': '2864ce6487be47511a67753ac54e54c8ac54185a90eb7584d76a0a9d581156a1', 'gzip_base64': 'H4sIAAAAAAACAz2RQU+bMQyG7/0VVc9TldiJE3OjE4cdKGgbXD85jk07rTC1rNKE+O9L2drjK8ePHztvs/l8cdCN7WRxNV/cPl5PX9b3D9+/TXHx6VTz7U87jNLbCCOuHlZxNa1vpxAixLik5VM7NRpFAgsKwDkwBVAWkVJ61KQGgRGIU6FqCMLgSdmz9pa6CSrzx7AL/45CJlq6HF4/tJIoVc5dnEMOsVfk2mPp1tNAUS+1cW+lFM+hJhfVWrICAyWmHtuZfv/17vFmfb3+fLP8cXh5PqGdqEVuQx4ikGN2YhfWUMLAB8vQWixVWwmDb+gNpXpEtsi1VT2j7bCb+n57tP3y159/zqlBilHjWFMgBBxaEUsDsJTIqrsYj6i51BoM0UAIeoaELFku4OO227PaxXi8PI1nzbWJJUtdRQDANWCppILjqON3THvxcX1Aoz4W1NjcueYzeHeUaW/SXzf7l99Pm//awWLB1mosINFKYdXWIRG0XInZHAN3RMjD2wPWcaSGjR3dCQHaYsDfZ++zv1dCz2RZAgAA'}, 'PROVENANCE.json': {'sha256': 'f66b19b62e2126f35f69fa9c070ded0e52bb178cb70777e3fb3a8f139e198b8c', 'gzip_base64': 'H4sIAAAAAAACA4VSXWvbMBR9768QeSgbNI4k27LVEkYzsi1jzWBdB3sS+rhyRBM5yHLoB/3vk+cudKywB8GV7tU5595zH08QmnRtHzQI30ZQbXs7OUeTqx+XYr0gVHxcrpfi2/L6/c1SfFheXq8Wqy+r7z/FgQgscOb2915Nzl5BEd1G0pINYIbWVSkJMXmpWWELZjhlnNlcgSVc8aqsLGaUawxEYs51biWUJbcg89oyeMb3ct9t2ija4BrnB+DlXQxSRzCo93ojfZMiG9odGqQj2CkwJj0FsBDAa+gu0PhZbtH1p8skDx0gOOvAjBx/suJIZmSEgYpiyqaYTylDb8ZOkQzR2USPhpqLxBKDg0OCjm4HKM0hPenWdzH0g8a3I0UDXkl/K4ZcMEd8wqafb9bTgWask71xUajebc2/IvhYY90WupR4TJd0XdwsyEKsrwTGhBKSsaxRx+wwwaMjwAijgDWlvMScYaq5lLKqDNGFBop5ngwqKlZDTiWnttDcltqowiRLNB/pR8zR9T5sB9xNjPvufDaDPrptl3mtXOa3u8y7Tda0hxn4GODhOT1LtkS9yaxu3Duj5r7XaSRw6sz8RQ+naa7xfg/zRg3hrjUwj3AXJ78VPJ29bP0rwyVjmZVdlK83XkjNal4aaTkuMTF1zmtDqrQlBR0Ws6oVN6qqKlviukju6rS6mqZ9LTgzRP2v8QBdzHrv9qGNWRua2XN8q2Z/iRvFnwzn6eQXEYSva4UDAAA='}, 'esm_driver.py': {'sha256': '4a4b2411c1cdba2003acc137b22e446e8ffae97b2c57880e33e2a62d52439a5a', 'gzip_base64': 'H4sIAAAAAAACA81Y23LbOBJ991dwsg8gE4q62Mlu5OGD11YmqfFtLCWzKZUKBZGghDFJcAFQsieVqv2H/cP9ku0GSF1s57JT87CusggSQHej+/QNz549u6qMkCXLQy+XCcvze4/f8aQ2PPVG44uBVzB9y9NOwdRCwDJPJ1KJcuGJ0mOlJ7TMGa49vX7vrXi5ip49e3aQKVl4FTPLXMw9UVRSGe8aXg+acaJXobdkGudD7zcty9CTOvQUDz1dzyslE67hXd/Dj+FFlYkcpoxiCZ+z5Pbg4OLqbHQekww/SHnb5boYUNMf0MOXF/T9zcveGTlIeeZptuI+ShLK+W/B8MCDP3yNURw7ERyboopxFK2FWVJdZ5m4s1ORG78gUcWUIYHdDaujtRKGU8PvjI/CR2ldVNoHDqEoU16aeBCCJuWalqyM37Bc88CyiRSvchDZMT6wEq6luuXKF2VVG2rPKWGQChUmLFnyUPGV0GChRvhGg0aqZGk1UupMqoIrbaet5ne/thtOaiMn8paX4neuQvt6IVOev5Hqwlr4/GJLYFkvwNgL1C5d1hsTvs1OKgFWKVmllxKElOsylyy1G0Fqp9RG/ODYyu++2WHzJSpuYRpUoEBTOp6omof8TmhD5a19c3quRBnb9V2yZ9wCpaatViLUP3GaybxSGq+dcepqJoBYZHloPxi2K2JrOzyA9nGB4ix1Rg2CKWlXkdmGEAdDbsni34aUVY0fRE46UWbStxANDeq8wUCkl2xvu0OnKMNPxG4kQ7dpy3wj7Odg/5BRVud5wUyy9Mm013nNOtns01HvM9kBjGJCc+8Dy2s+Ukoqn1iTb6T2ilobb87RkfkdS4x31OskSwZuZrjyElkUwnjjtycN9FvDx48Q0Bx2o4524DCMcIi1UX5jT4AUCXY1E+5oxTkOOAiIUOp4Sp47G4dkBfFpHpk7A+PnkWYZN7zUUmkyCwt2R50n6XjQOGqL9ngP+xEinFaKg5eIkqd+e5jQxj/rgprKMr930DQKlAR4KyS4fAL6a4xpeVirxU8505/EJqw1pztH3XEQx9v+Rkb6JKlqEkR8xXIfgg3Gh0hzQ8u6oGaJ2NZ+IUr/KJQ6grXApC4B6p5UXj9odxSsrEE6zUHiQW/wqvd68NqxS3fdxTr1NmQFe75zrPk/43RK4FHzMuFkdlxJjV/gITDb0P6caZ7ChOIZTsCDK1xLGWscDpAOBKawpdOf/QCIyp7AM2Qob7MXRlqkNfcKoa1nNLBVcg04clQhKnprkZol5i9gvIaADdNktvVrOx2jquwozHnpgyR4LgNpIAao+b0Q561sHbuq2x1sFrovsD5RsorxEHbj0P6+sJPbmGLRECMlO93pb2YsfnW8gbGP5MDFTK1K2sKBVOANLE2prngiwHLNri1MrC7Tho6eEmc1+LIT2EDX8AHDU8WnqG08CrILXgwQHzA57TngvujPIkh+hR/8sBUtSmS54so03OGBDHywTPAgXj5pQbur21qP5WJRFpAbvIyJvFac7B1kR5AdAbBKcdyB844pwdAO2BCSG4Shx/gP5MrlQkAuss7kP3/ujhFE7vMOx/1NVexowzKqZWYQGW5LmIoi7vS3gq9NnFfT71FYq97NVoQsY4hXcnJ6Nnrz09t3P59fXF7/cjOefPj1I9k/Cbj/93Bi7DGj1lciVlW8TP1PjXegTw6dLxAEBbVIha+tGzfI7oePHHwIH8I9Bnt/hIEbScoSAUsZC4krNikqFIrAOZuLXJh7MoRThQS/yjTVdKXpuf3WWZvPW/krqEuNQ5QTHDNYlUPoTIckdAfI8lovd7zDIgRLli7uoxcn459HZ3R8enUzGkdQpkJElaAMn6xJWPJ1DqE8JiTwmPaynZARw9LoTCTmV6wMlZ+FmeB5WrKC6ziHusNHxU57M4gKa1c+LiFiwsrNOy6wq5xgbLHYjVnfBwBQCUS6u+kDVc0siTukgAzQ3WHNju5ncczYFtzAe4uBRybirKQPTKHrwkfeQRcDhx09bXUCxICWklpD/ikxYWgyxFjqNhHM44/nwa++RhRWatAxhAxaytJhcMUf04HSwl/9GPesOlaojq9Rzeoyca0RTXKmtchEwowtysjl1YS+u3wzurkZnZHPT0Lp9OpyMvrHhI5Hl+N3k3cf3k0+/qmAAiN9G0+wyAmHzRbs/PR5g6gKz2/zeFuZQBZf5HLuk+ck2K+eI6Ftnn8YNpdx08Rh6hi8fPUgmFiNVM151fzRKXcj3BzFsUfNWTFP2TCzNYXf7x397eVfXwXhHLQ0XEZ1lUK/6c/3ObnjTbHGxCYLWlIEAFRFm7MFs3gZLfldKhYctNcUtFh/byw2npxM3o+bavMTgaBmakANOb26uD4fTcDS4TfL9JCsuVgsTVPfObWQoZNvAzRi8wYZuvRBKQRo3ExpSHZ7N1iw87a/LgWGUFoNbdkHcRmy5lxt4Xk5IltuEBa5wlK0nXdFqtcWaJj+PHRnD+vRY+gwPDavy5TBXOhBMyBWEIVD2zjLBeSVxL6C1SBnJ5Cy9/0jQo+wvW0BpW+LGazn7iG3qMUKSow4Jp2OK9h34lfTCz/fLBwMZ8GxK3kckbLC9vhBaxxvCQ9nu22S7x+G/V7wo13QKg+bs+lwMPsxxtnDYAfTrjy5gbpYFG2B0t6P4G2Ix8uVUNJVJ7qusCvW3vU9qKX0DqN+7z//+jc8Do89KNu9pFbY4nqnMmdz1NbN+0sKVFxl31Q1QFFJ2bTO7VVHtOAGx9gnB0GXFCtGoQmmWLav+rRHe67lre7jZn+XzEXZrawk5LhgqMft3M3o5Ozj41bZLds2xtvSU93ve+n2UiZSdelPUZ/umojNcx6SToH9GfAjIfpgwziYhWCm5PZBNcrvEl6ZXZqnEJl5eu3erN732f/FG8uCN4ps1D2vRY6IxRYVChuoEytRRd7fgS+IwCoPLGWglwFZbCAC2HjQ25Z4seHBsUtohCuW3DKIB9Eet3lL4ttG2fKgm11gIfK/Kw+Eh19RQtTJcxh1OqnQON+BmU6D3Y5Vp52FigvkceresA6+UmthINiKG8eDXjR4HR2Sp23UQDOGXrHBPBSS1T2kF3idkuuPk7dXl9cnk7dQO+yL8Adws5HqIXpQAvh/UsJHpDH23wd/UJ3YpwuuYTxw6oUAAIiB8WGPfFmrLpCDMqNXUc9uxAvAu06tkOnSmEoPu932liQC/7QRX6pFd73Muxi7Z//HpyvrAkIMnG4AQNnPTXF8FB29tofeuS3sLOt5HDfI+grh3RsNWP7SkW97Fvtt0I/6X1IOpm0XvCBPI26GT4XIvbqsvZoMXK6/Gf3y/t3N6GJ0ORnDy/jq/MPozN4tbeuzh+XK99gkU5z/zkFubVK8Ec0eHeBLVPBphaS2dqAUL1W0zOGoAfJos2X4dA7c19QBRHhKsVqkFFItpZiJKW1SrUvLB/8FuTIa/f0XAAA='}, 'evidence.json': {'sha256': 'e33139e9c58bae4e4dcaa222fc03786ca34de012ecd7fc4c23e6db62c1bff985', 'gzip_base64': 'H4sIAAAAAAACA91Zy3LbRhbd5yu6PJWNiyRI2bIcc0VT9FixRKkoKsrMZArVBJpkC4/GoAHBQCobV1zzDS5/RrbeRfyvObcbICnJLyXaTKpsUgK67/Pcc2+3fv6GsQfaW4qIP3jGHhz9MHAno8H+9OXk+OzvL93RDwf7o/Fw5PYetGhlKi6lKITv+jwTtGGnu/Ok3f2uvfPULtCeSsyLKU8XIhM+S1IZ8bRsa5WnnmA892XWYrHKmC51Br2Z9JiVy1SK5UrNGf7xmRaxJ2qxGbYJDcH/wq+M/Ww+6YWR6kqfdL4YjIcuLNp1B9PB4dlkNDa7zUKPx75srOYZD/NUxJvXCxGLRsTAoc8X5nO4WRIpX4S0JoHJIs7avkjlJTwMyyhZqlnIdaakzzwRhppBHZvLWWqf642YTMBr3/UUnIuzFLJU7HLtpiJRKd6QhsPhIQUiEzJm4nUCPRHWPmM7nV0jeJflR33mKxNGMj3loawEy5ZS47HGT4qJS5GWjGvNSwY52VKwhEPWxhSFEKfwALp1HmakeQLrlDWKkjDPw7AdiniRLRmFo7FK95EyP/fgvbdMVaS0iniInInUbrYBiKSXqjj3QiGNCcb1Ou/bIUl5rOcidUMZyczsJ1v25RwP4bhxscUueSo5fkPoMvE6syp8qQXXom9C8fzsee85LNNeLghMXihj6cEwMZ/j2ys3Os1St3lOeKYCmPNQi9YNcOWpSfsyyxL9zHGKoujEPAN+Op6KHJ4CwKHQjn7ce7zTbQN+9H+v1+5utH2ubsyaX1pfh+ujw9GPg/Hxj58CdhSK1zxWrz+L7D+I6btheIdASln/OJg/i8ITCirSZmxeb06FpwjTlGvz6sWtV1soM66ZpeI/ubzkIQFpnsceGQvZM8RlLjOqlNt08AlETp+/6jkHr161RaJliBLRcgFZMl4QwoQHTCIdjIeZSJnO4JCGRb5IBD7iLCypplLB/QxFky+WFrV8g1vy//8SoqdHw96AMPrkzpRrtv4xSDKEXtwRl93Obov16OOR5adFyiMnOuyzncdsSQACpSQhL6GSqLSNtCQgNPHV4F1jckOl+DUmGSTG2Nxnc6gVeinnmX0Ctb5ti0tVgNp4qBYq11sQt2ZsmBcBVPAAwnXGZzKUWcnWnn4JykO1FBpiZ1JBUQntRLYaZqmI8Djp9ZkGp4cm0G0ycSsA2nbwNb9ayr0P6Cb5LBJ+J/ZmshOHUSeWy85CXTqPe3t73Z1ez7kvxK6pFKjt7dydTyOkWyahWCe2HeUIr83LNijXgF7mEcjB4PZ2C/wK5J4KampyIYm+aCKyTJjHKI9U45np/DK6iYCPYHTc2Ey9lCOJYLBoMh6wGkmVha0ZYrZGgAaLuowzgo/+eryNj/bXHMlT0TTx9poc+yzkRJtmzDDerSuibujGQD7LY58jSjepXt8H+nwlOypdOL1up9d7+sTp7e12273e3k57r7379L6gtz84IarsfgJ0O60nbV/ySMYqyVMU3m3wnf190IQ+dRA/RVwAsqCA3AbXGoEGe5RUVD3N/KZ2zYR4RyiOVd2zQAlKg9c3LGmi/TnsWdvXPdDYk6VlkqlkyYkqsYN09tmL6en3vTbsU540lRUJD0ukjr4It8O9R3vnDAAl/JxPWShyD5HsQ/wl/JMLXo8BjUg6hQDeM2GZTVAhhFIvoRWO5yncxI84H8FWwbPrXeB+ANd99NR06Cd76M6A3W7vcbdd3Rfohi+mE/dgfDqaTA+Ox0R6e5/A37XkLIjtbwOQpH0EYJsOgdCu2QIDGB3lajqxE5jpcCUo5G7AG5A9Tk0eTCfCkwi64UHKnm2DOCmyBhfpF5hwQGXW5h6oZo08JomQqJ3ymrG25kaPUG4d0jfGuS+i8twZOhMThlv0xw4tLTaJIHfQ5jNtfItwxkGV+TecnOOMY7B371D87pGzjBaO77/ufffkLhDE57/NkX3tyUdO7V+YDGkAU1quR5Xj8f4BgXZw6E5Gp6PBZPjSHQ7wcH8wHW2hcM0ONIVNh9cn7TqeGwDmcRCrIr4ZOmIOt55niOmOp+7RaHB6Nhntb5aaVu7m8UyhG7mZuetwM6l1LtyGEg3XjV+Nj8/HLkn552hyvJHQrHIjbJZxo2o4OByeHQ6eH47c84Mp3DybTEbjKQr35Gx6up2IRY5+qdLS1TFPMDMaNI/OCFm+rLHK82yJiaHu5raVxpS/Z2w0ZD6KR9OLHXMgeIQk9tlS0tRaD3WNDtboaNFJaym8AMcngE/YMgsbMvVCLrcbz5aVKnW3qNa1ELx5ZhER74g8VQl9oa85GAUl8bZ2TMSd0clg4tQ605h/HTA3LFePqy4dFyOwlN9UpqmPj3Lol8bB+8DqZINTh+akht4acgB1JiG6+63jJxbOCYEY3/5SMH65AaEdmnMMFk0b6DMvT821EBAY5RZ8LXaBUUkDLevB1Y5BaOvc8+ggTnb9MJocvDjYDsFdIHof0+H9gfEzY+INUB6Nhi8H44PT6cHQPZkcPx+5x+PDf3wCjBjP2plqnxM9KEJW306VdqJq2PMvBTdMszxBf7isJzyJKHuWM7fnQHOJipHSXGGvr19DPhM0RG9ENPMiHapClPafB9ufmQzvBrh1+95Mcm4SchPpv7GTVGHkCrKyiEtGj1mqKp2lVbmI5dU7TOkGJg+ZY9ttkP3+W7Hdhn+Kf4qnym71Q07BoysS1Hoe9dnqAwfbsULiOFsxXSHyKs5ZLAW7IAgWZYA2E5cRYr96X0jue5XA26hDcnsd9vDhiVWLAw5fvQlUUT57+JCBN2BHzK/+i6NAy5pov87t19B+TVjB8mD1hvsVVAYqwvHezLLehYQKxOe8XEtKK+FVZSFX7z0sriIYXan46i0DM6/eY4E1gfftNFl6VczX7/Zxki0QSDi/cW9ztI145i0BH6v0FeJbcNbrOo+6zm7X2dsFKLrf0rHG7EOoLyBLRC0ja1FwNEjvAqbkVakDQIwz6MfxXVVSRXCgdoYlMLrQlbiAs6G4iBU5w2bcJwcTHgsa4quSqUCt3ihmtDaeioUy0Yy5zq7eJaQ5IEMRCx6K1YdamPKt014dwTHWKT+tcs/osOlCjjms81FsSD4bnR5hvqiYCIC2tKQU5B12Xl//6wA5QOcrce5ffSB13hKJLkieV129MwEGTwXIWt/oJNsqbhJLiyq2+P03DDUX5IK9/5LIDv1WiZTDzh2CEt02yG3wEpSUT9gIjXKceSDLqjeoDKhvB9R1cDIJaXBZ5Kahh/kMOxMF5/zYQgsWBJLVR/dY2OCcJikvoABG8jAUodKFAj8IBltaLBGrNwQwf/UmX1zDGOzMEAvNI/sYRFzh9AJId9gpj5hHeGvuC0w4iNLzC4T6+2EbFtZAg/0pgLBeSkkpZAxva78N8LeOh8iumAtb5kghN9RgbHDsTVLcGFQLrP2ETUUFMqHLikUMH3LEyCtxYuKAGxlYoHvz1KtQP4kCw4iZsQIllzGIUgsaOPGojgDkPqKkDXmu1/eRpvYJ4irDSsQ6UTI2N/6z1Zurdz7mWrEAJ1N9g7IAEbX61SNsExgrklxagydU9YjMIhTPGDJbzfjWcltbdgMWttjqPRUJgsATlWSq4i0gEy0CZ0FVlfiFRmhrjGShCuivdzCCswBtPFMAVApI1VU6vXaTfAGtoD9wN5V1YDPGTZkiNjFtY/vErCKgpOg1osiE0mYLwY34gl/LpGnjLaPRUlJWhtCEage04WtCf0CR3DF/1cgNorWAEgqdhbMIYPBjQ8JE73S+l5Xh3wugHrkOgF0AIZSBCbjhLmiq6aLcLvetsiz7m2Q468JBDs1hvoAoKm70HwjxVx9UU0snoJ0MTMoUGomnDD0U5QUxkQJJpWWFboLwE5/FNUwjNDG/KjdWYrrqsKfbVhdNX5IEEmQKpoEFAuQblJMb1ZSMi6t3GTFKwcbPwdwvulBFe3xVmC2R8krTApEyWmfYNqfLp4LgBKYALlICrgGXbTurX03gt/JW+4r68Jv2iH6IgETozMQjBiGgaKAuwIEtAFdvCbccLbfCDRNzQJWDp4hiLV585WFH0RQDp2K2EYVdyJ30IBWMXVSLEBEESbXY5mfI/273WzY82GJLMhJQIu5Pt+JNTWKXIDQCFlAqyIBhXa1mMcBYAB5zGVjVollCHqnQKpVXbytu+ANBW30oC7oobuxHn8sCarB1K1IRXaWj2qmDoZVQRVWM+A4xBy0bVoMsTaTLiwBoMCmzV59Qc+2pV6FHIrSGjJ6QE+hYwATgwWlEiqgRkzezlAeWzGFQQ+bUd5uBoe6KH12DtghOQVGX13scQqFS00XpJSeo89V7SiLf0o+obxGVY3nKQ9D6daZt1Zumj9UbJMCKdby3OrkpHmmyUbOraeWD6UmLhKAkS5qOiHkM0SPbyBZ11gTTM5jXjJJoBmUsg3yLjs1EN4SZtjAtbKhhFSamqiBYodumSSpoPoNUugLPyiRHCmH6tcE0w+o5r67eQiz1YwoW2ssF3bIJvwzMsIR53lsWMpR4DdZLMXK8REM1YGW0i9or1b+a4cF6astyCKKU1WB8C+o1YynB4ME3v3zzP5LHGltjIwAA'}, 'mva_readthrough.py': {'sha256': '0e173bb8172a1e779ccbd2462b58699ef309d33252c5f038dedb3b9f3ff6322b', 'gzip_base64': 'H4sIAAAAAAACA7V9W3PbSJLuO39FjSccICUIIiXL7qYMT9ASrea0RKlJypoetQYBgiAFkwTYuIgivYqY6NjZed/diBMd8zPm7Zx5a/uPzC85X9YFF5Ky3TuxClskgEJWVlZmVt6q9OTJk7O3DdYLbWfM9lj7de3ZDn49N9hF0p94zk7oDt3Q9R2X2b49WURupLOLRXwb+CyKbX9ghwM28fqhHS5Y4E8WRqnUDtjI9YOp57DIns4mLht6E3rPmXi+59gT5vkKaBCy0J0Gscvu7NCz/ZhFSX/qRZEX+EbpyZMnpWEYTNnMjm/RC/OmsyCM2QUuxYOBHbuxN3XVE3WtM/q9DHy3JJ840Z3Obu2I4OBLPJ3ozIvdMA6CCXB7FwW+zqYArAMjnUW3SeyhSbTAwxjkcfsgkc6SxBvobOnNaEyl0ttmp9s6bzOTaTWjalS10utGt9ml695R40QrNRpWr/H6tEl33rw5Pe3i5/vvt7aOjrauTvFzgZ9vvvnuuw5+Wq3WWQ8/7fa333a7nc5b/DTwc3zcbJ7gRysdnR+ftwn6wHPiMrAolzXNeBd4ftmpsCGo6YC42biMWRgMEjTlWNHIZq4dm/uVis4UZpUKkOQYN46Om29Ovml9e3rWvviu0+29vfpeK3Wab7rN7+h5+8yqVmt7tZrxXCu9bXRajXYP999roI8fOaE3i7U6E+11pjnWLIi8GDOJu3t7tSrugZ1wofU0fLcn1Fw70fQSy340oBy7np9/+cX+C/GuZdv0yql8XV5urUCIgiR0XHpyG8ezqL67O5/PDd/pe4Y/mRq+d2uMgrtdYkdw3W7n6G21Wn3+rLa393xXeyiVBu4QlPox8UK3HGDOp24U2SO3Uue9eEPmBzELxvW009D2Ipe9tSeJ2wzDICyrNwSs6NYuEwdLALegmWREA4/2Dp6XK/zB3ItvOW+L1kYwc/2yFva1CrMjNsz6o4nuq4kuT+xpf2DX2dAIXXtQrlX3nrEtRh+Y5b6mVep56rBbI5mRmJT7otfQjZPQx+1b937gjdwoLku85yHgc1wgAu59LAHRDQwhQ/SQ3zJmNmQ6NqbjgReWxUVk9sIE0uTee1FsBWN+KbqNpzMA4S/SuC3fnoq+DPrGtplGAGMtbW1wdCxCpEy/ANV3goHnj0wtiYc7X2lAhNqBySeQV4GbGMkgmc7kQDB0Ww4kPz6Sf4OaRWVqQcCjJHQtO3I8z3xjTyIMw/MHGJS5pzN7MgnmwNkXjyqE7w++VlHMYw8sgpifdkln3tEksAdROTfV/A0+tLVBSZix3Z8oZMNgDq009NzJIDLbUHKqC9wHUScgdpm+C9qJdrgvv4B5ymmT6+pNhViav+piKOz6RvGFkIA+FElZvAp+0o66b1nk3LpTm3E97Y/kFG1gi183x5z9aQIF388h5L47h5C6pqZtmO0VoeCzGaI/aHrjGPrxit8oDyWlqMvIlANZeUvw1i0mAS8A8fxdokyOmo9w2NDGclgOXScIAb4w40pDa680DFmNHNwiP8TTyP3x2qt7219Vb7h8eyTf0Ksjt1zVJy5vUNG/qlYUrxWEOqcYqAOsW+6PHIBAyMA4plE55SXS1xPSAaBKiqyYbuqK7rKnbJ+ZJiOtfXTcZbg/wvx4Edd+A+/OizxwJOsvWHwbuq5W5JrIjQWYlyaj71hcTnoaMVBj2vdGSZAAUOJM3CD2Bi6hik60yia6iUXvGtA4gfZvHiEQdafvqzGCsSLXSs2XvCTCrDh2oX3dECSYLFg/SCDYA3bi+q9tfyxeDevA453rxBFLoApmZESgjXsf+LucHoFj0/IUGWSlENiowP05kS5SRjvvtE5abY3Qj2BVDWg2d3fFNejTAp+T1QSbSCEkZlFSx3Ycl9tH6C50jci1Q+e2HGp/krbID9F2+YfudgUiE9GKb5xVjFEYJLNyrYhIDpCZrdqXvns/w7Ax1o477IKN0na7d25InxIR0FuikPTL2vWfbGcU0yzf0OpMvRvRbOLF6YD1WuW6dqNuYsR0A+oH69GMxE4AdZ0Jlo4BIHs+VFR+gKfnR5ddDA+j44McbFfw2Z9peqQXx1nZyNKmqaADQUVbCErCTVHJ4FBqsAKdW8WL9lwMcgjlD61PeHBR4zQmNK7/9IN/g69+ufy7evl3vxGPf+hWrn+IfujeVLZ+Vyn/zlR3/+1Pghr56RFaGoYZFpzIvL4pKVEeo0+dOA3reDBYCIafZypvKlFLJhOOM5ATVDF+MPgXTcfLRhTDNCtXMqUHfU+gCV6ZhFvTNeLrVUtB0W+qpJ6WGs4imTxItFNxgNG1TbC3NXzBzUKn0yJ8NWTDBgP4gzIMycUMdhsftgZ9iqWhTlwwBd9UgKM/UNd7uN6kAHNWIBAGLKAhtAgRUPJepfIg0BILCoh4fc/pfU8UUVgRwvfXAqUb0+R0ulnjKwECrFUDYZpKckCnnD5zTNEKzH7IHCzZpMqcaznCm51aHRc0uBsBX1rAZqamHbVkqa5lEwNvRbRulmEEV4Qy2dL4XGEkstF1fad2wxXLnT3xBixz6VQHmUzHZl7idnMNzCfl68bOH3+IwOtPNB0YcwKvGAtxnlU4PlI/hNoPEdhM02OaS9NcwU1JY65DrDGwobGyRMIyyOhJvBqZEWfA8pdMHWfuG33sLkxhLLP7+n1K/3VlwTuovKrxAfALTF3aHnOdewD85eSZZroKFbQ4tcMLoEqe0qkWhk6xJYTtmmn20374uGydG/rk6vH+dNFrrX5DPpzWSz0vgQ6DduP0dwI/Fsustj4+YifT3K8d7POB0C05H5UdoFCrHlSLK8Hry9e11znGEbqyuFy/1zB0rY5fuoYOtDp+6cqb0+qFCacGlhTwTBCKEs3bcJFX0iG0FCALSmhqurW6+iadN+iTycKK/Dsaqc6UJ0nqdki2e7xi88CU8SmaQWaCags9IwSq9tJUN1+ainogT7f9NoXMgiSOyI7JGzEZvdFtRXAH9a34h8wiPNmmey/NnJEk5Wb4GxOP0NO5DyMF0hB7vhOz43aDoe9I+CDuYKU/YHet0AKpTZOPWeukc5ezulbXOjGRBKGeA0EIbufB1m8Emf3pwBq5wdSNwwXnTpou0DuKgxl9taZhal6+S4C7l5PbtH2Fs4RY/IM55gGG09TMi3X6LpfrVwXwN0UWBMvGlmpvEc1oVi0ffDbFIpB1UNkpgCHI2UPhCPHVrsiSE1j5VtbusZ7s+3+5pxmsFE/AHsUHVSxk7bNj66LTPG4d9ZrHGoGx/UX5vgj91UE1o1sOB9GP1j63jhrt83brqHFqNX9/ZBWBfhKHgy/H4eBfwCGNv0h+texkgDXb82cQMSW35qp1z+1u2WZX4+rKyoWrjFFf8nhiKrfiXtlFGbIbwZw/rx48f25wB08rmPXCnCAHFa6VigsRWGgzuSq/0lZs0QQCeZ1qxRshmbCydy997wJ3M3tULX9T11ce8jQBk8Vmpt4AilTtjS6jcdf5oFvuLoXdcpcUeZPLnm2b5Y0v79Qqu7v72zVlKIEJ+MDMsm3j2RYeHcr7mHgz12J7T7jzscVvmgrJ61wbsnnUuzCKMDDZWAzx0abFpdM2zRf7L7i6lE6i6pRW/VMt/yTtgh5tkYvyzcnbrrKRaL1MKb+mFkV36YBMrddrCOAZ4rh50tCKyyZ/wjwKGnnxQoIryIokjlwLb7bTke4IwpN6NQt69vperoYZ2wKGWBhvbvSi7hVTTAJkvtdE1AZSfPa2YZ2CcBacvman2T5qWjXYZqk0aXXpDWqJ7xGB8I6QgpVAayako74lQplanaKdXyKOm6O+CspKgDQxePjHhe2ej1MWYcgUglaXDC3MDGGrcMXM12LJjZXUNlEtKJhMLZIVsOn8cJpatb4dYdWtF2cOfWUsazlaPXepntG0pE9wUexGsZfFzSh1pWspi4kH6WX+SVh4ksanKM6uXa6SusAi6XAKd3UtDrG0wfUY5KlDkl+ENXDJxB1kmhjfYAglbgRUEkXzpLJj29s1Yb/lH2ScW9G3tojHi+CJ86euTZFYoCiisNoMvEGe3WW70+yen76lhUuLb93QnrlJ7DmEgpO4Fqb3zvXlaw95OyHUhZrRuWiIBccJgxnF2PLm4twb5MK3QgusmYM8Lse0I7xfsAdZMBSRpSiNbwKaSWYI/5YG9SqHTGhWshuqOjXIrK8d3nZ3dy9tLu5UCiYbBQ85jLrQwLzJjS7ggvApYjv8jgyKB5SjKGfJG2sY2o4ylG1ACINkdJu7i5f7Hpa8Re4euTlgAUy9JBSczMi83gR1I9ANMDOQN2uuEmXoDC8aej7F7u+FrVx9ad6/hFWdakTCAQL+RkKMoKejmPUpE0nv8ZdogUbT66peu5Ez9Ft2Dq8V7BMHFIJCf2BvMUQek+J+aO/qnAEhPFzA54KNFEF+2FVPNo+M/MRUjYMtjjIl4socLUH8STAqh0EQr2aXptHIVElMww/mZZXHNJLYqWDkAQYJiOUKj7DIl7P4OQe6q4lkrRcZ6EdTmSRb0x8JorOhiHWX0f22SGSIQASPtog+9OEkiW5lBoeP4c4NvaFH0p9wB5KipToZQW4/oLwZxC8MsfqZaQ5KNjJFmFRckEkVBZM7V0YAYUepF7Nw0Uy8oh6svkOWZa4xzLvYDc1cGkZ0tatw29UorAJbPkp4cNOgVlpltTsJCPZTQiZ+CDW10nEh5E1pvl2tc9m2zhrt1ptmtyfhmmYKaWr73pD3LBY6sgAvZ9IZUA+Zc0tx7VXXbobph3Kj8Mida8VBSsECDMnpkr3xK1Jkh3xMBm6ojMk8fTaiXeh8KgI+RD0KdajvwgriLnycRCJQdn52cdrk/kSGFigo4xIiKqLGlmYsxJJOwgtovGBAu1Fpi1zm0wSu1D43EYcKxeEafaADLv3IHrosWaMxRem19VnkQKj/stAtNKvkvQv8dO0oCMNkFmcAqS1FPTlWedmf6VPpJ/drz6SwC5uIVIo/9EZqaUmXI/MRt0d4x5TEFOKdGW6Ny+OWnC9d2ntCktcato+tbu/8QjoyukhYvZfG2VXPotBFPXUo5H2H8vY960Q8lWjmlmhq1qnhdf5q5tbI+2Rl9tzQyoX1rL6LOXc5LsWXrusv9p/fPFTyQb5cMNzTeZrI9ZMpT9zkjAe9lmMRfi8LKfPYX92DtUTWjjDPlM3jZqEn8Zjsn9xDEXUqFSzelLuIJCqGlQHaWTHoa/raOzympaCvt893R1E82/MjKzVts55ervoOe3vVr1+aCrAMcYu0sWCG5h/OScYvDCe600QgsbISnLl+r9lDKCorJVsa4hD3OZn63ADbRCBlFqpwCBXpWKlpKQcgBm5fb7ZEN8LLQi4iGvJ5KK8Oqp8FdPBlgA4eSpsSrZ9gSBHlrFVu1ufg90fWcavba0Amu2IiUuqrpIRP4YU868+yVCfQLDoaNzv7VT29K1yMm+39vERIiJlM5CtzZroG4T9uNzLhv57xGK2QdvFIuuX8waN5Fw2+FIm18ELgLqzhCtv5pbmKLNwCqDqYEYKtZqZJSmcD/3LYR+ftXvMPPUE6OTJlvTXvb20sf5A1no+aMCpPmLg7BBhamdKXuMk1K9kpbJTAlhA5WFwRFpQOcodDz7GdBXMmtjeV5hwM+mJuLhgOIzfOJma/slKfQ33iqYjpFhNfMHHosUmkKLp3N9cC7k2dB/D9xC28iIYmL91Yf7FyiAv1tkng6UYa7cL3SjG5x0e0mSfWJ25bANY1Emje4BG8P5mV0+JAqxNqpBSjZBIrTsFHeoscTRmzwe3PAJS2Onn5ieP5buFVHgPSszbuPdwBWuqyXvkUFDzvz3RITBKEFv5x5rGG8E494cBQfPS8ZzW63Sb+HWubOLjVPjltWlSNZ3FErc75ZU8pAjEla4s9FWVa32D9Pn/zRq7077e2bF3ZXHWt2/zuMrMGLGWAfTK6+xg9MxdNyYEc11mz0b3sZOOKkunUDhfmUNO03/LKUfbPP/83my98b0wytogTRuv/VqkkCwk8JoOcdRETWzhL34bvOp7D4nln6+y9jAmkJkHlgdn2IWVV0mciePPA/NgoXQTx3HPDAflHdeYY3Fh5daKzMQ+8Ocb7dU5++Oef/yv/QCqhhzrb2ur1Guyf//GfrHfSYOXLk0Zla8so9YTtAgThMbMZ+ptHS3cUMFJH9BasFmC5taWzedB3HUbpM9yAS2iUSucDIPPx3x2bNycIPgyvDz/PfAKx/PjTh784S5vAEEqPpy9uaMRbW4clQAho4n0vDyAXZ1ewvihXIcEapfO+RypylHz8acHKlEbwGAXy/biiAK6mJG4eOHlAEzxYjIlAWN12wHRQroA2C6aejWVyEnz8m+OJ1O44XMzixGBtPF/a448/2QN7Ck1rf/xpHMBloTH1Q3ucsE67AfJ9C8EdxzYPZIDq79gY07ioywmm4G8c8Imvfs3nFRxQ05nNgsHEHX38CT1/+OtjRCdWhcoDFIHviz3OVFe+F4An2Tt7zBEP5guGMQbwyn0xEjAtBolbS5qJdy6LXZ9KnYHv1tZFCDSpmUv9QiwO2DtyN5Z2ZA98l4MV4oE1YGDPneVCkGO+GNtLG+tQTgaxNHlMhLOMra3SGzQQYQYv1ztwD4MlZnm5GPneh59j22DHts8fsET069nvPvyFTVxn6eK7UYLMbvAVupenPat7eXbW6HxvTAeaLmW8kvoxB5/yY5RLpFtmITQgGnGxJjfUIhC07JPO4Gs4nhojyn9yVUee6PnbZqfTOm5SyYB0COycv6p62qQc4Q/qlg6+/6QzlcaUrldDz1g28PL6bW4moLkyxdEujTNz7Zelx1cSCDldfnnR7XWajTOlxwX+IBIFYMN0XPBbVwMFPK6ejXuDu/6gRsZL/sz3ZNeRAl51zXjo/1H3KzVwoMFgvjQa3Xq+hMe2f0Oral32cs2hadu2fWOuA8Pt7cJdtK3frDGeQKjRu+w0ei2siRed816z1e4WXVVVWyndwztaQhw3xxTFpINqUIhlBPOCDbdhiJH7468biIKbWVL21PMDy3a8AUXMKfhPKjhxKI0CR1lQUeSnREZUkVDXVjIRPLS8edUGLl4QUtg0DCa81h2z3ceAbyGw4/VOyoWlXW4LII+UwmCyOTdXr446Cq+IiiqE0y4NbYu2eGiP4rSpi7RUzIIsBRSmt6JFRHEdog7v8pT6fAymCCQM5eK1aovACJJZu8cyR3i+OXe0wTwD4x1fHvWsi0a7eapssnnRLyswEI+z0xiu0ZV+UNvTa9Vq9abATrrIAXGHxORZhoI40t4GEdFf9ddyLMUbFNlChAFwL5fB4bkmMmhW726nyYPaJ6xAIYjKDaD5InRTSHwE/zrBi1XCP15zsLxShVvrlEfZndoRVb3yTNKqNi14090z5RB2Cx7huhVNTVvti8teqnrVQFb1Y0oANXCaolyuk6aBozr3fNhX4OPi7D8od7TL96HkSr8oEHvvTSG1VCYH15MnSMhdpbWFKufIR6X5xwclkGHcUJgyGGHVdvCacEZ91w77izwfDmUaOK2Iuiko7aGqhYPuBjUdS7bTaC/AMIsoFQNK8iEPj7ySj0SsZH/dO5W1AcPHAmGimOCQSgXKw82xr1y9AZ87aabytJioBKCJwP8dyp1WC5WlqvFLE1NQFwTKxEeJyoqMaHUClAWroA35cqlu6Fquaq8+VBWPGxRHu9novP4+l09/06TlTDl2Ah/9OkMkRWG9+0KvskrDjabm+8zfIxUIA4D7tnbEtSLZA2BxsR1Fe1CZlLxxRUkMAMpX+f6WtajwDRZluIvHPpgPXBi5M5sCWgNd5JRgk3o+FDcFaWiL3GwSxLQbj3bEYRjEJ2M39N0JmNUZ2yOVA+M4iN13UdIH71OqRYf7Inbn6SzyRr49ySK3/p0ZRFAed14Y+IYTzBblCljm7lqDQ/u6cfRts30Mu0trjEaauN87x73WH5udLpR2p3F62jxtdc+ozZATIg/7WvvmjfXN5WuKv9HGN6vXPG2eNXud76l9TUI8PreIvL0OuhP3c3oZ1jAY+BoLmOHeu07CeUAny23F+Iim1iCEYIfGjNZLarFRDWWPNi2qKeDNFjQ6mkLNTix46bcuYOVnWzwJXdqnQfoOdnRmsMThohiRyucOgXwhbTgXWULcGtbXcKRZNbPZNS74O5JSGNwgSGKTXqXvbhjm23Z7x+eXPZ2mHf/FOmn57tyKxH6D3G6xIlVI1fAE6TSApATQjeAT8ndNkd5efWNtuOmwb72Jy8dgzILJhKq4Il6NV390leT9RhPXnZUPKo+2guitIrhDCL4ynx3UPxmHSRPDxCrME+LJlcFuBAOYApSitHtEmedDmGr2jBLQ2rbgpAQab61rTpYKpYzxRuS6LJ3jT+NCJF0F9qvGzDt+tfcMSvmTPYkdlD28Dn4Reyi18xmN2p4wIgQAjaiU2nFd2qrzrAqzwH8EfSDCp1Tk4cgS+Y1ZrYsuOolPSG7sIuVM2q8YS4quQKqsdzn0fCoGqH8Sly9lL+i/sTeZzEai45kHIeJa0ui2TnrNztnj9Ccu5y/NbTi7saAl1sPH3yByzvLK2VAzcD+DeTaofxKbb1unp5XDrMdiP7RkZb5Zpvy6PayM3dWEe94oxJuP5pMzVvDSAvscGDmgJv8gg8umuXTqa5jlFtM3jdZp89g6vyDfs3FKCJ40c0srMQFgwOK2lQvSzSToEGhTbpsHeniKjInENAmq78bzIBzvDlwyRGA8ejDNismkFQs1Tx0dXeTKG76oZTFsfLAaNt5QAyh9ni4vAVytwtsYCMktVCJ88etDFTpfIdUUFCY8D136kFHqp9ZFvOFKE1GMI/nZIbNduYj2xFK+/5q3mIOtDiOwKK4wnWJ+pJ2nqsyk/8qLf2IZ4rewolMMFIBb7eMmyWOr3eg1Yfl9d9mCvWe9uWwfESOpKPlnAmxp8PyAB895p2O4B8HAWVIA/fKkUSpdLWD/0z5Uig5e9XTVjEXj8Je/Y9GHO8FqX7OZ+/En333HBh9/SkYiVDkLBiDs3HM//jtFDBfOO4p3vjBKFza31yiIsIjdmIKcgwD/Pv5t7tkDESakwC91x6P44uNKfByJjw4FgK92j3Y73FPZgDwL+pEbzgl1l80htP7CuWWJCPw6t4fMwyVehJHovKMWYqcJRS6jD39hSx/voQs0oYTc0Zteh13V9r7a+0MOcABTfPzxH7QrJXEW/oef61SNxYOuc+/Dzw4DkeaA+PEfbBp8/MfEmy+mCll3ij5n7pyjNSbfcQx6ObdGaSVCPV+IqLEMAYtBU3SWQv7ypuh/oUZxKLMcQ2/MsweyywSwpvbIlvHxxCjt6c93Bh4PG82ScOHbbB6N7WVCsV2bXYmQsZrzqUvFQfAZKHuyEJFrn4AvF3PiBpduSEyMUgmqYmePR/ALUkahe/Z79+PfJh4FniE1BAuzQKSijqHa2EXZtv9NEaXC5svR5MPPA5AsG+g+C8Y+p9c5PokLqPLMo+B6aHNapyFzBUii7LlLe4I54QV3swmnkWuwMzJbeYtgHBJ+NrPBU3NJYh4N0kuiaFDe63uUPbBJLOz5AKw0CPrE8FE8t2XwnM/UGObMgjWOzk5EwD2KgwhDxeSSNZWwi0a3S4jyDFZCJgEP6F9B5BcgPPTL2HlH+sVf1NlnxJ+i9T1eZYR5YTGEMrKn+K2SIRJnqtbyWMQRX3of/grbDGbdCKT48DNzbsNgGkTBFHOi48p1xryMDMI+CcacytTOKB0HDKs35kNkJeJw6fZxxSVAiBo42bkN5sk7sDDJ5yyAShgQTeSkULpll7YzOFi3ksMSsA2YQ1uN3qlRUP6D5zQobQHtkPGzGEwwd0cB7a4SiYdgAPKHyzQnxHMOWSLh+acSCbPas/+dHMKsdvArAB+sAT74kuQEsN+cl+gXGh1sWqJXjl643rzAwirixYbU0cbVlS8ofXcS+KOICyUbeEMev4pFphYjL1gucOGpmoHH2chyUYWWq7sHN6ZL+htshFzlX5oW2bCj5XOpEWF+YaByUOr6QFw/Yns8SpjPbBnm8/EIxINP5Vz+R9mIQiDzLfiJAkf8SB5ZoCXfIa2dDMh0vFl/8Qjc0zomHXTcPGrR/v/u6tsOPHGPaoozAGlxtnldNao1Hb8O8Iu+7NGXA71mVFXxdRrT541F6+Ir+L9/szHDEuthrEf6cPOpRBke+dpw/VqAFUjQVbWIVy6AVcy8bCg6t+BtUcmHKM7W6rGubSpD1+ph/FgGIorxwoJbq661Xq4OBikYn1n1eq7J8DHg9gDurJV7XVTlZ/j3rs4z/GEsq6p9SdhHczERVgQb5p1I2dT5OTywuuEW8l0dZNNCDGmvxIy2PqltEuscdnzehUNkdY+a7Uandd5dzYv8lrUGLq1GFC907/kaEk8WjPaP8JPBKGrIj3gQ2yHYCIrlkIKKWJIil0qU060b/CiR6DaYDEQEMbobmdfaS3yw++nEj8wndKKTPNBpvm8E4Wh3r1qt7qLFE7nB4slXL6pP2K3rjW5j88lBDRd3njt/HdybT6qsyvCY0d1XBXfgZUgF0o9CGMILNp/Mb8HDT3ZX3hyB0/14ZwgDbrIwn0TgwZ2Ilhj12m9rL/b6zwarPVKeggGpfcDHe/t7TwSgCKQ0n+wBwwYxh9ynwcpPaUcJ7T7I9hpUuNegpjriJ7C93CXAn+7r4EWhr9oz9BVFydSNsj0ezGRin7JiTroByynIznHLZuuTne69OOC9fl1d67WTq3WAVuirvolb8mcB2DFM9VmsuskUzDvIwUoVZgayUgdfKPUwzBB6v7d3sP1uq1atPnDEarWvi5jtP3n1Hk+3wrg+eniqeq3kapFXOs00T045be5cTsH72sHX297Ws68e1rqGLcZ497x3mnU6TCGd9SI6X0iIgpa4oyzJVl6T1PQa7JNJ4pq042Pv2cFOba+6dbdbW4lxFofEZYbTs1bN0/N9bX9PjU2K1NfPMol6tq8kIxz1y++p2wf9Pe172js40OlyG/0+6HsH1QpJ26dQyE3piyIKm8kLnnt/Vzf216YVYLdJ16xM07Pn+2vzc5ZMYm9G239gVTPHnUzoFCS4B34w93MClBceSEu2MLBy9Z9//q9aZZPYrAvrs6+er6HQxkD3vuWZ4tDrJ2Ir1tB1B+KUwtiDNMPgcieUgxCYcDwj8I8QZnVQBM8TuKkEv9wd4T+IoYQsH8BYWwrQDkZwdobV3UjawyNaK4UxIIf2XqN7Wl07oUDTjwktRTyY1oqyyj0gCh7eweI3pbC8OxWHy1Bh9u+0XMEjuWpWWvV43j79Hk958ZzcLchNzLqoy+FlnzkzlBcDaQ/6GmJ7a4jZA9rpFbsiBJKzm5h9Z3sTWiULeF22v22fX7VXCy83odbgsrwTzVwHzohDDhiz+1gfKRdIggweD4NZ6FH3fJI2oby/hvIICsCDuqBDgXbkmUbca4ZxPyhiu1JJsQHJ85CSz5x71UbpAViYc5AMb3jDRZqpt2MKLm3C81kRz+MAKw3Wfcpv8Ez3jgIhTUPaQ2OjN4G5EqJfiX1DkXOH+zV8QzaFjOmxLUSGlCpkg4e2KKSzCfeDDbhLP51LkjflO035mpVub5InpOExpdZoOyhPSMp4xK8cyBnEOetQz2ADLhwM3HD5uYBZjGDTOJ6v8YoLn5DT2ubUwfTNJnYiznVLfGGtxXYI73dHahTYdwEhtonrH8Fdkl69uaMsTk54xVV9z+dHMT0qmy/WkCf2cajIng6bpfIGxjcGs7TgaAXH3GbhDWhe8PMq7ckuPyOJ26wSHHyvxSEbu+5MdMc38s2IP/nsQwUPPKHhtYf1/R7KK7NOGmmdd6oc1ZFnlFNRXpMdTqE0y1QppsN1S2jHQlbXVdHLufo9XZsEUbQTDHdSG03sVbXjIEzbXnFA0gcsVJdKYSs6xuU0tP65l8oy+P7Zlpm5Frq5vSw1/euVTRNpqZxOVW1cC4IcK6eGUeFeXlFe16r6flU/qOovDqgC52Y9uyZInDmKfS+YBCMZ/iePhXMZvhaq9TJsNFFkR783+VuaEBIrRcuCLnDIvaKNceldXVN72bOWYL2NpW9aqkksJe0Rb0sFPDllYLl3dM6qeGTPglkcRLSBMfU8HwOfqgqLpgyKRIAYY+2IA3hmoWvxyixvaaeANECGhvB9iCoxsHArNxfSdS+aR73W26b1pqpyTNZxs9s6aQshEFOyniRp/uGi2WmdNds96+K00eZZkiyWQUdrhB7JLEdEBTSKaa/nX5D26nUa7e6pqDet5RXF6haKc+t16/z0/IQfVwMdcnTZtI4bvUbRZsuns/rX+Us6GQJqw9p4asFnklD5Hrg5cAd2oCEIl3lhyRXYy52OkD8IIfXPrMwV3wSb68Modmc0/teX7WPanWadNXpH3zSP09lTsW2+lZTT76jTuuhZjbeN1mnjdeu01fu+SJVULVip3Ykejs7bILFItrZpC0yj/S26AVmajc7RN6L68tfmz55zT1i6jM7inb+QRfY5hVQq/ZGU90hm0pajYEDnGS7nMJGH3lgmqSj674koKf84MCgVUGfvV+b1wSj17L474XsvgsEvf5+zYDBYenRHAhngq0hoBQNxj1oly0U0tnnGiRrInBhWMTykXNfSFSk79vH/hr/8feB+/IkH1Kc21oHlLPj4t5BC97STgXecsGicwCZTqam8Bk4R2OEGrNigwDN/czuEwxLMFzpBJYU8psh/EVZ9awtGgT2ByoKZw+zpBLa0H9xTFkDu8BlCQdtjoU2XtL3icEM6iyewIIoDm3KC//yP/7wy2Mf/B3PUB/G9D39ZumORM6A9DNOE76gQTLwQQHluxih1Oen5NhU1yUt75kU0m7wP8BrN4oKfTS/p9/Enm3aEEHw7iQOYghwoJUrE5iGeMYMzzDeI8HEblHPlSbFA7juixfpBBVqSaLlgzhI4upg4vE0p02CKWayLBM+Hv3Jz/pf/U5gMXGZ5KzTBtcgp2Zh1vmsK/EgbUiLvw89E4q2twfyXv6MLmcCUmTDiFrkJ5Kq3tXXI5iLH49uUR/JsTrBZ8M4dLHyH71TirRPiIGe58Kmn2u6eUboAtcWruA0fi916IIorMot1GoJZqz4tHAYibuSHQaEE3JIj4VeYBkzh1hbW5KfgoIE9HxOqu7QhihJv4G8+y5R5s8nVi2mAbImWfHBSImgR+uXvfTCez7fHLH3bWdoiXUeLK084ydwd5xcPE3e1DDGTBbpT0hfvzz7+tJjboLvcvkP6QpJD9DDFvH74C1x5tFEGwnIlV0l/3kBuJaLjWPlZHIIRYQ/O6iyhci2P7zlK+QlXub1PGI6gFeY4dTtEmnC+4BNt++uJQvD+EusrDLyvsnSpJ5NzAEeJbFoQifs9+1Ak0GO6xthAqHcyL7sANZwF1wuUlBwB+Iefl55kQI81eheUR11isAvHs9kk6cvM5XCS0CZKh7bqkZLhNnZoy7mhOi9MDm0yAuPKrUyeSr/m86h1yWVLvGpzRlHsThUeyWFeA4FyHkuASu7GmC+ZtHFQ7YBCnzRDvFRgayuXZqT0Hi8W0lk+uyeTjnSghw4uDWZi83XxmB9VZMTKMsUoE4LcvtD4uRJpuGfkpuelrB9vwrvMn1YhckJmlh8iRFbPFaGXTHVqCe9it/yJY2Fgpg7pTll7+v3T6dNB7+k3T8+edv+oVbY1S9umP1ph0K9nZV6Rf13/6ibrZ9OJ/emB/eJ4e7UBYuiNzPfZOSB1QV1NnQtdl0dQ68Vsal3W0YmzSlKKP6yabjLZKow2mRA+zJ96cdkulj3lqrjwrN1qn6SbbvMVn+o8tlx2TkzCrsZLcTeee/LI4SQK1ifOKOHRRF76b6p+HjuxRDRbPbZEZg3Ts0vcad/lmYCVI0sK/JpCWjm7RDzIHWDSIugiVBAlUyzgHq3vxRNMRF6nIQ8Rom3ncosCx4zHDkWAxqV7/ORTlwcaxR5uRucHYzBZGbj4cyq8pDsOXVcOURfzKi5yA0trYDl/UdUqlZG6g0M2DAOs17lgnsRIpfRzQFQCmU5fUclj2sEoRZi+P3+45j3ciN4K2CgGfByp1EM75F8TERCQgV484LEhcRpyPsWVbqvOQBd5vAfvJ2XwGf9bOxChRWRIMYN0gVdo6RG31ZWuQRdo9S87Perhsd4flbCsDDP3LhcC8z0XcCPPw1zrVUTuvJIdYiEPf+VqJxxNgj4/uJofnjlbYd6ZESUg1f1vTPlXIh4ew7govvoXKagNw9KlSNf5R26QvDhe/skf44/e7I03KbokXQNPaUOQpqfNWhdwd9/Aw2we8/L55XoU41MUWY9hbCQRiSwIlEejvpRnec30TVNSJGJxidl4KhboqY6/qmebFTbvT31sTop7v8QZfGhaytXsvoZ7/Hjd7hfyqKjlpd01VN+d1e4+rPzBDxVr6HTOO3yrR5N2lxjxfazp6R97MoSsWACQPzz/E9wgS4n/daZY4YYvZIZMXn4NH4hD3iTuhwye2wheCGkqilFN3LqmPz66/JlwhVr+UgkoWvwvvliWaWqWNcVyYVnyJBS5LcgOR/wkX2E9merSaISjhII8F/wvcWDBnBn2YGDZ8nZZE1aXDhfBc+hclk1GmjRxVl/d2ZFMj3GJxXMgBnC43lKtBZ9tCAnZob1Vugi7mRo/csSK0Vz92Y5wFJkzQxxcTBcYFdXnk5FKl4ZQVfyrOkuPX6ilUuzeMvk9eZE/pa8MYJtCJIUTi6lcnxrn2oqJrJT+P56AvF8KbgAA'}}
PAYLOAD_DIGEST='33eb7f05cf165b2e143ee7e08a679293fda1fea195731e20b126807c7b09bd35'
STAGING=Path(tempfile.gettempdir())/('mva_readthrough_'+PAYLOAD_DIGEST[:16])
STAGING.mkdir(exist_ok=True)
for name,entry in EMBEDDED.items():
    if Path(name).name!=name:raise ValueError('Unsafe payload name')
    data=gzip.decompress(base64.b64decode(entry['gzip_base64']))
    if hashlib.sha256(data).hexdigest()!=entry['sha256']:raise ValueError('Embedded SHA256 mismatch: '+name)
    tmp=STAGING/(name+'.part');tmp.write_bytes(data);tmp.replace(STAGING/name)
spec=importlib.util.spec_from_file_location('mva_readthrough_'+PAYLOAD_DIGEST[:12],STAGING/'mva_readthrough.py')
analysis=importlib.util.module_from_spec(spec);spec.loader.exec_module(analysis)
print('Zweryfikowano zamrożone referencje i kod.')


## 3. Analiza i trwały zapis

In [ ]:
ROOT = analysis.run(NOTEBOOK, PROJECT_ROOT, STAGING,
    run_esm=bool(RUN_ESM and not SKIP_ESM_FOR_TEST), model_revision=MODEL_REVISION,
    NB14_RUN_OVERRIDE=NB14_RUN_OVERRIDE, NB15_RUN_OVERRIDE=NB15_RUN_OVERRIDE)
print((ROOT/'RESULT_SUMMARY.md').read_text())
print('PAKIET WYNIKÓW:',ROOT/'RESULTS.zip')


## 4. Raport i pliki do przesłania

In [ ]:
from IPython.display import display, Markdown, FileLink, SVG
display(Markdown((ROOT/'RESULT_SUMMARY.md').read_text()))
if (ROOT/'DOSAGE_SCENARIOS.svg').exists():display(SVG(filename=str(ROOT/'DOSAGE_SCENARIOS.svg')))
if (ROOT/'ESM_CONTEXT_SENSITIVITY.csv').exists() and json.loads((ROOT/'ESM_STATUS.json').read_text()).get('status')=='COMPLETED':
    import csv
    rows=list(csv.DictReader((ROOT/'ESM_CONTEXT_SENSITIVITY.csv').open()))
    display(Markdown("### ESM-2: wynik sekwencyjny, nie test funkcji\n\n"+
      "| Produkt | Średni log-odds vs L | Min | Max |\n|---|---:|---:|---:|\n"+
      "\n".join(f"| L737{r['amino_acid']} | {float(r['mean_log_odds_vs_L']):.3f} | {float(r['min_across_contexts']):.3f} | {float(r['max_across_contexts']):.3f} |" for r in rows if r['amino_acid'] in 'LWCR')))
display(FileLink(str(ROOT/'RESULTS.zip')))
print('Po zakończeniu NB14 → uruchom NB15 → NB16. Prześlij RESULTS.zip z każdego etapu.')


## Interpretacja i źródła
- `COMPLETED` = obliczenia zakończone, **nie** skuteczność terapii.
- `NMD_PREDICTED` = reguła odległości od złączy, nie pomiar RNA.
- `NOT_RUN` / `FAILED_OPTIONAL_STAGE` przy ESM nie są wynikami negatywnymi biologicznie.
- `INDETERMINATE_REQUIRES_FUNCTION` = potrzebny eksperyment przy wyrównanej ilości białka.
- Każdy run ma nowy katalog, manifest SHA256, źródła, log i `RESULTS.zip`.
- NB16 sprawdza, czy NB15 pochodzi z dokładnie tego samego NB14. Po ponownym NB14 uruchom ponownie NB15.
- Model ESM pobiera publiczne wagi i działa lokalnie. Sekwencja i wariant nie są wysyłane do API inferencji.
- Nie czytamy WGS, VCF ani danych identyfikujących osobę. Analizowany allel L737* ma publiczny rekord ClinVar.

[RefSeq NM_001211.6](https://www.ncbi.nlm.nih.gov/nuccore/NM_001211.6) ·
[UniProt O60566](https://www.uniprot.org/uniprotkb/O60566/entry) ·
[ClinVar L737*](https://www.ncbi.nlm.nih.gov/clinvar/RCV000641226/) ·
[Reguła NMD 50–55 nt](https://pubmed.ncbi.nlm.nih.gov/9644970/) ·
[FANC 2025](https://doi.org/10.1038/s41420-025-02571-0) ·
[SMC1A 2026](https://doi.org/10.1002/epi.70150) ·
[UGA/DAP 2020](https://doi.org/10.1038/s41467-020-15140-z) ·
[Produkty readthrough CFTR](https://doi.org/10.1093/hmg/ddx196) ·
[EMA Translarna](https://www.ema.europa.eu/en/medicines/human/EPAR/translarna) ·
[ESM-2 35M](https://huggingface.co/facebook/esm2_t12_35M_UR50D).

Zakres testów dostarczonej wersji znajduje się w `VALIDATION_REPORT.md`. Test rdzenia offline
nie jest testem pobierania wag, instalacji środowiska ESM ani uwierzytelnienia Google Drive.
